### Kernel: Python

## Treinamento da rede multitarefa (incidencia e latencia) com covariaveis selecionadas

In [ ]:
# Configuracoes do ambiente
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

SEED = 2003
EPOCHS = 100
BATCH_SIZE = 64
LEARNING_RATE = 0.0025

np.random.seed(SEED)
torch.manual_seed(SEED)

In [3]:
# Carregamento e preparo dos dados
entrada_rede = pd.read_csv("../dados/entrada/entrada_rede.csv")

# Covariaveis usadas no modelo de Cox (vars_paper)
COX_COVARS = [
    "stage_paper",
    "prior_diag_trat_paper",
    "hormonioterapia_paper",
    "educ_paper",
    "quimioterapia_paper",
    "macroregiao_paper",
]

REQUIRED_RAW_COLS = [
    "rhc_estadiamento_clinico",
    "rhc_diagnostico_e_tratamentos_anteriores",
    "rhc_primeiro_tratamento_recebido_no_hospital_hormonioterapia",
    "rhc_primeiro_tratamento_recebido_no_hospital_quimioterapia",
    "instrucao",
    "macroregiao",
]

def prepara_covariaveis_selecionadas(df):
    df = df.copy()

    # Se as colunas brutas estiverem presentes, faz a recodificacao
    if all(col in df.columns for col in REQUIRED_RAW_COLS):
        stage_raw = df["rhc_estadiamento_clinico"]
        df["stage_paper"] = stage_raw.map({"I e II": "I/II", "III e IV": "III/IV"})

        prior_raw = df["rhc_diagnostico_e_tratamentos_anteriores"]
        df["prior_diag_trat_paper"] = pd.Series(pd.NA, index=df.index, dtype="string")
        df.loc[prior_raw == "Com diag./Com trat.", "prior_diag_trat_paper"] = "Com diag./Com trat."
        df.loc[prior_raw.isin(["Com diag./Sem trat.", "Sem diag./Sem trat."]), "prior_diag_trat_paper"] = "Outros"

        horm_raw = df["rhc_primeiro_tratamento_recebido_no_hospital_hormonioterapia"]
        df["hormonioterapia_paper"] = horm_raw.map({"Sim": "Sim", "Nao": "Nao", "Não": "Nao"})

        quimio_raw = df["rhc_primeiro_tratamento_recebido_no_hospital_quimioterapia"]
        df["quimioterapia_paper"] = quimio_raw.map({"Sim": "Sim", "Nao": "Nao", "Não": "Nao"})

        educ_raw = df["instrucao"]
        df["educ_paper"] = pd.Series(pd.NA, index=df.index, dtype="string")
        df.loc[educ_raw == 0, "educ_paper"] = "None"
        df.loc[educ_raw.isin([1, 2]), "educ_paper"] = "Primary/Secondary+"

        macro_raw = df["macroregiao"]
        df["macroregiao_paper"] = pd.Series(pd.NA, index=df.index, dtype="string")
        df.loc[macro_raw == 2, "macroregiao_paper"] = "2"
        df.loc[macro_raw.isin([1, 3]), "macroregiao_paper"] = "1/3"
    else:
        missing_covs = [c for c in COX_COVARS if c not in df.columns]
        if missing_covs:
            raise ValueError(f"Colunas ausentes para covariaveis selecionadas: {missing_covs}")

    # Mantem apenas linhas com tempo valido e covariaveis completas
    df = df[df["tempo"] > 0]
    df = df.dropna(subset=COX_COVARS)
    return df

dados_modelo = prepara_covariaveis_selecionadas(entrada_rede)

# Split treino/teste
dados_treino = dados_modelo.loc[dados_modelo["conjunto"] == "treino"].copy()
dados_teste = dados_modelo.loc[dados_modelo["conjunto"] == "teste"].copy()

# Pseudo apenas para treino
dados_treino = dados_treino[dados_treino["pseudo_incidencia"].notna() & dados_treino["pseudo_latencia"].notna()].copy()

# Sequencia de tempos para avaliacao
tempos_avaliacao_teste = np.arange(1, 132, 1)
tempos_avaliacao_teste

array([  1,   2,   3,   4,   5,   6,   7,   8,   9,  10,  11,  12,  13,
        14,  15,  16,  17,  18,  19,  20,  21,  22,  23,  24,  25,  26,
        27,  28,  29,  30,  31,  32,  33,  34,  35,  36,  37,  38,  39,
        40,  41,  42,  43,  44,  45,  46,  47,  48,  49,  50,  51,  52,
        53,  54,  55,  56,  57,  58,  59,  60,  61,  62,  63,  64,  65,
        66,  67,  68,  69,  70,  71,  72,  73,  74,  75,  76,  77,  78,
        79,  80,  81,  82,  83,  84,  85,  86,  87,  88,  89,  90,  91,
        92,  93,  94,  95,  96,  97,  98,  99, 100, 101, 102, 103, 104,
       105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117,
       118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130,
       131])

In [4]:
# Preparacao das covariaveis
covariaveis = COX_COVARS
covariaveis_numericas = [
    col for col in covariaveis
    if pd.api.types.is_numeric_dtype(dados_treino[col])
]
covariaveis_categoricas = [
    col for col in covariaveis
    if col not in covariaveis_numericas
]

estatisticas_numericas = {}
for col in covariaveis_numericas:
    serie = pd.to_numeric(dados_treino[col], errors="coerce")
    mediana = float(serie.median()) if serie.notna().any() else 0.0
    serie_imp = serie.fillna(mediana)
    estatisticas_numericas[col] = {
        "mediana": mediana,
        "media": float(serie_imp.mean()),
        "desvio": float(serie_imp.std(ddof=0)),
    }

mapeamento_categorias = {}
for col in covariaveis_categoricas:
    mapeamento_categorias[col] = sorted(
        dados_treino[col].fillna("NA").astype(str).unique().tolist()
    )

def normaliza_coluna_numerica(serie, stats_coluna):
    valores = pd.to_numeric(serie, errors="coerce").fillna(stats_coluna["mediana"])
    return (valores - stats_coluna["media"]) / (stats_coluna["desvio"] + 1e-8)

def gera_matriz_design_one_hot(dados, tempos):
    dados_ref = dados.reset_index(drop=True).copy()

    # Preenche covariaveis ausentes em bases de predicao
    missing_covs = [col for col in covariaveis if col not in dados_ref.columns]
    for col in missing_covs:
        dados_ref[col] = pd.NA

    base_num = pd.DataFrame(index=dados_ref.index)
    for col in covariaveis_numericas:
        base_num[col] = normaliza_coluna_numerica(dados_ref[col], estatisticas_numericas[col])

    if len(covariaveis_categoricas) > 0:
        base_cat = dados_ref[covariaveis_categoricas].copy()
        for col in covariaveis_categoricas:
            base_cat[col] = pd.Categorical(
                base_cat[col].fillna("NA").astype(str),
                categories=mapeamento_categorias[col],
                ordered=False,
            )
        base_cat = pd.get_dummies(base_cat, prefix=covariaveis_categoricas, dtype=float)
        base_cat = base_cat.reset_index(drop=True)
    else:
        base_cat = pd.DataFrame(index=dados_ref.index)

    tcat = pd.Categorical(
        pd.to_numeric(dados_ref["tempo"], errors="coerce").astype(float),
        categories=tempos,
        ordered=True,
    )
    t_oh = pd.get_dummies(tcat, prefix="tempo", dtype=float).reset_index(drop=True)

    return pd.concat([base_num, base_cat, t_oh], axis=1)

time_levels = np.sort(dados_treino["tempo"].astype(float).unique())
x_treino = gera_matriz_design_one_hot(dados_treino, time_levels)

y_incidencia = dados_treino["pseudo_incidencia"].astype(float).values.reshape(-1, 1)
y_latencia = dados_treino["pseudo_latencia"].astype(float).values.reshape(-1, 1)

X_train = torch.tensor(x_treino.values.astype(np.float32))
y_train_inc = torch.tensor(y_incidencia.astype(np.float32))
y_train_lat = torch.tensor(y_latencia.astype(np.float32))

train_loader = DataLoader(
    TensorDataset(X_train, y_train_inc, y_train_lat),
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=False,
)

print(f"dimensao X_train: {tuple(X_train.shape)}")
print(f"dimensao y_incidencia: {tuple(y_train_inc.shape)}")
print(f"dimensao y_latencia: {tuple(y_train_lat.shape)}")

dimensao X_train: (48610, 22)
dimensao y_incidencia: (48610, 1)
dimensao y_latencia: (48610, 1)


In [5]:
# Definicao da rede multitarefa
class rede_multitarefa(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.tronco = nn.Sequential(
            nn.Linear(input_dim, input_dim),
            nn.Tanh(),
            nn.Linear(input_dim, 2 * input_dim),
            nn.Tanh(),
        )
        self.head_incidencia = nn.Sequential(
            nn.Linear(2 * input_dim, 1),
            nn.Sigmoid(),
        )
        self.head_latencia = nn.Sequential(
            nn.Linear(2 * input_dim, 1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        h = self.tronco(x)
        return self.head_incidencia(h), self.head_latencia(h)

In [6]:
# Treinamento da rede multitarefa
model = rede_multitarefa(input_dim=X_train.shape[1])
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0

    for xb, yb_inc, yb_lat in train_loader:
        optimizer.zero_grad()
        pred_inc, pred_lat = model(xb)
        loss_inc = criterion(pred_inc, yb_inc)
        loss_lat = criterion(pred_lat, yb_lat)
        loss = loss_inc + loss_lat
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    if (epoch + 1) % 10 == 0:
        print(f"Epoca {epoch + 1:4d}/{EPOCHS} | mse={(running_loss / len(train_loader)):.6f}")

Epoca   10/100 | mse=2.773861
Epoca   20/100 | mse=2.767073
Epoca   30/100 | mse=2.763863
Epoca   40/100 | mse=2.762968
Epoca   50/100 | mse=2.762025
Epoca   60/100 | mse=2.761827
Epoca   70/100 | mse=2.760641
Epoca   80/100 | mse=2.762176
Epoca   90/100 | mse=2.763380
Epoca  100/100 | mse=2.760580


## Calculo da sobrevivencia total a partir das estimativas de latencia e incidencia

$S(t)=(1-p)+p \cdot S_{lat}(t)$

In [7]:
# Predicoes para avaliacao
dir_saida = Path("../dados/saida")
dir_saida.mkdir(parents=True, exist_ok=True)

dados_ids_teste = dados_teste[["id_paciente"] + covariaveis].drop_duplicates().sort_values("id_paciente")

linhas_grade = []
for _, linha_individuo in dados_ids_teste.iterrows():
    for tempo_grade in time_levels:
        linha = {
            "id_paciente": int(linha_individuo["id_paciente"]),
            "tempo": float(tempo_grade),
        }
        for col in covariaveis:
            linha[col] = linha_individuo[col]
        linhas_grade.append(linha)

grade_predicao = pd.DataFrame(
    linhas_grade,
    columns=["id_paciente", "tempo"] + covariaveis,
)
x_pred = gera_matriz_design_one_hot(grade_predicao, time_levels)
x_pred = x_pred.reindex(columns=x_treino.columns, fill_value=0.0)

model.eval()
with torch.no_grad():
    pred_inc, pred_lat = model(torch.tensor(x_pred.values.astype(np.float32)))
    grade_predicao["pred_incidencia"] = pred_inc.cpu().numpy().reshape(-1)
    grade_predicao["pred_latencia"] = pred_lat.cpu().numpy().reshape(-1)

# Incidencia deve ser por individuo: usamos media na grade
pred_incidencia_por_id = grade_predicao.groupby("id_paciente")["pred_incidencia"].mean().to_dict()

linhas_evento = []
for id_individuo, predicoes_individuo in grade_predicao.groupby("id_paciente"):
    predicoes_individuo = predicoes_individuo.sort_values("tempo")
    pred_inc_id = float(pred_incidencia_por_id[id_individuo])
    pred_lat_interp = np.interp(
        tempos_avaliacao_teste,
        predicoes_individuo["tempo"].values,
        predicoes_individuo["pred_latencia"].values,
    )
    # Sobrevivencia: S = (1 - incidencia) + incidencia * latencia
    pred_s_interp = (1 - pred_inc_id) + pred_inc_id * pred_lat_interp

    for tempo_evento, valor_lat, valor_s in zip(tempos_avaliacao_teste, pred_lat_interp, pred_s_interp):
        linhas_evento.append({
            "id_paciente": int(id_individuo),
            "tempo": float(tempo_evento),
            "pred_incidencia": float(pred_inc_id),
            "pred_latencia": float(valor_lat),
            "pred_s": float(valor_s),
        })

pred_evento = pd.DataFrame(linhas_evento).sort_values(["id_paciente", "tempo"]).reset_index(drop=True)

arquivo_completo = dir_saida / "predicao-rede-lat-inc-completa-covsel.csv"
pred_evento_out = pred_evento.rename(columns={
    "id_paciente": "ID",
    "tempo": "TIME",
    "pred_incidencia": "PRED_INC",
    "pred_latencia": "PRED_LAT",
    "pred_s": "PRED_S",
})
pred_evento_out.to_csv(arquivo_completo, index=False)

arquivo_sobrevivencia = dir_saida / "predicao-rede-lat-inc-covsel.csv"
pred_evento_out[["ID", "TIME", "PRED_S"]].to_csv(arquivo_sobrevivencia, index=False)